In [19]:
import os

from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

print("All imports successful!")

All imports successful!


In [41]:
# Load FAQ
faq_loader = TextLoader("faq.txt")
faq_documents = faq_loader.load()

# Load Travel PDF
travel_loader = PyPDFLoader("wanderwise_travel_guide.pdf")
travel_documents = travel_loader.load()

# Load Hotel PDF
hotel_loader = PyPDFLoader("staywise_hotel_booking_guide.pdf")
hotel_documents = hotel_loader.load()

# Combine the LOADED DOCUMENTS
documents = faq_documents + travel_documents + hotel_documents

print("FAQ documents:", len(faq_documents))
print("Travel PDF pages:", len(travel_documents))
print("Hotel PDF pages:", len(hotel_documents))
print("Total documents:", len(documents))

FAQ documents: 1
Travel PDF pages: 4
Hotel PDF pages: 4
Total documents: 9


In [23]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

print("Number of chunks:", len(docs))

Number of chunks: 5


In [24]:
for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)


--- Chunk 1 ---
Q: What is the return policy?
A: Customers can return unopened products within 30 days of purchase with the original receipt.

Q: How long does shipping take?
A: Standard shipping takes 3 to 5 business days.

Q: Do you offer international shipping?
A: Yes, international shipping is available to selected countries.

Q: How can I contact customer support?
A: Customer support is available by email at support@example.com.

Q: What payment methods are accepted?
A: We accept Visa, Mastercard, American Express, and PayPal.

Q: Can I cancel my order?
A: Orders can be cancelled before they have been shipped.

Q: How can I track my order?
A: A tracking link will be emailed to the customer once the order has shipped.

--- Chunk 2 ---
WanderWise Travel Guide
Sample Knowledge Document for a RAG Travel Assistant
This fictional travel document is designed for testing document loading, chunking, embeddings, FAISS retrieval,
and question answering in a Retrieval-Augmented Generation (R

In [25]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
embedding_deployment = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")
chat_deployment = os.getenv("AZURE_CHAT_DEPLOYMENT")

print("API key loaded:", bool(api_key))
print("Endpoint loaded:", bool(azure_endpoint))
print("Embedding deployment:", embedding_deployment)
print("Chat deployment:", chat_deployment)

API key loaded: True
Endpoint loaded: True
Embedding deployment: text-embedding-3-small
Chat deployment: gpt-5-mini-1


In [26]:
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=azure_endpoint,
    api_key=api_key,
    azure_deployment=embedding_deployment,
    api_version="2024-02-01"
)

print("Embedding model configured successfully!")

Embedding model configured successfully!


In [27]:
vectorstore = FAISS.from_documents(
    docs,
    embeddings
)

print("FAISS vector store created successfully!")

FAISS vector store created successfully!


In [42]:
retriever = vectorstore.as_retriever()

results = retriever.invoke(
    "what are the best hotels?"
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
Travelers who are currently on a WanderWise trip receive a 24-hour emergency contact number in their final
itinerary. The emergency line should be used for urgent problems such as missed transfers, serious
accommodation issues, lost travel documents, or major itinerary disruptions.
Non-urgent requests, including optional tour questions and future booking changes, should be directed to
regular customer support.
9. Payment Methods
WanderWise accepts Visa, Mastercard, American Express, and bank transfer for most travel packages. A
deposit of 20% of the package price is required to confirm a standard booking.
The remaining balance is due 45 days before departure. If a booking is made fewer than 45 days before
departure, the full amount is due at the time of confirmation.
Prices are displayed in U.S. dollars unless otherwise stated. Travelers are responsible for foreign transaction
fees or currency conversion charges imposed by their bank or card provider.
10. Frequently A

In [29]:
llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=api_key,
    azure_deployment=chat_deployment,
    api_version="2025-01-01-preview"
)

print("Chat model configured successfully!")

Chat model configured successfully!


In [30]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

print("RAG chain created successfully!")

RAG chain created successfully!


In [44]:
query = "tell me how to reveiw policy?"

result = qa_chain.invoke({
    "query": query
})

In [32]:
print("Answer:")
print(result["result"])

Answer:
Here’s a simple, practical checklist for claiming travel insurance (based on general guidance — check your specific policy for exact rules and deadlines):

1. Review your policy first
- Read the policy wording to confirm the event is covered, check exclusions, any deductibles, required notification timeframes, and the insurer’s preferred claim method (phone, email, online portal).

2. Notify the insurer promptly
- Contact the insurer as soon as possible (many policies require immediate or timely notification). Ask for a claim/ reference number and for any specific forms or documents they require.

3. Gather and keep supporting documents
- Commonly required items include:
  - Your policy number and travel itinerary/booking confirmation
  - Receipts for any expenses you want reimbursed
  - Medical reports and itemized bills if you had treatment
  - Airline documentation (cancellation notices, delay confirmations, boarding passes)
  - Baggage irregularity report from the airline (

In [33]:
print("\n--- Sources ---")

for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)


--- Sources ---

Source 1:
Travelers who are currently on a WanderWise trip receive a 24-hour emergency contact number in their final
itinerary. The emergency line should be used for urgent problems such as missed transfers, serious
accommodation issues, lost travel documents, or major itinerary disruptions.
Non-urgent requests, including optional tour questions and future booking changes, should be directed to
regular customer support.
9. Payment Methods
WanderWise accepts Visa, Mastercard, American Express, and bank transfer for most travel packages. A
deposit of 20% of the package price is required to confirm a standard booking.
The remaining balance is due 45 days before departure. If a booking is made fewer than 45 days before
departure, the full amount is due at the time of confirmation.
Prices are displayed in U.S. dollars unless otherwise stated. Travelers are responsible for foreign transaction
fees or currency conversion charges imposed by their bank or card provider.
10. Fr

In [1]:
import subprocess
import os
import json
from fpdf import FPDF

notebook_path = "main.ipynb"
pdf_name = "main.pdf"

# Load notebook
with open(notebook_path, 'r', encoding='utf-8') as f:
    notebook = json.load(f)

# Create PDF
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()
pdf.set_font("Arial", "B", size=14)
pdf.cell(0, 10, txt="Jupyter Notebook Export", ln=True)

pdf.set_font("Arial", size=10)
pdf.ln(5)

# Extract and add content from cells
for i, cell in enumerate(notebook['cells']):
    if cell['cell_type'] == 'markdown':
        pdf.set_font("Arial", "B", size=11)
        content = ''.join(cell.get('source', []))
        # Clean up markdown
        content = content.replace('# ', '').replace('## ', '').replace('### ', '')
        if content.strip():
            for line in content.split('\n'):
                if line.strip():
                    try:
                        pdf.multi_cell(0, 5, txt=line.strip()[:150])
                    except:
                        pass
        pdf.set_font("Arial", size=10)
        
    elif cell['cell_type'] == 'code':
        pdf.set_font("Courier", "", size=9)
        code_content = ''.join(cell.get('source', []))
        if code_content.strip():
            for line in code_content.split('\n')[:20]:  # Limit code lines
                try:
                    pdf.multi_cell(0, 4, txt=line[:150])
                except:
                    pass
        pdf.set_font("Arial", size=10)
        pdf.ln(2)

pdf.output(pdf_name)
print(f"✓ PDF successfully created: {pdf_name}")
print(f"✓ Location: {os.path.abspath(pdf_name)}")

✓ PDF successfully created: main.pdf
✓ Location: c:\Users\balaj\OneDrive\Documents\vs\rag_faq_project\main.pdf


C:\Users\balaj\AppData\Local\Temp\ipykernel_18760\3984464465.py:17: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", "B", size=14)
C:\Users\balaj\AppData\Local\Temp\ipykernel_18760\3984464465.py:18: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(0, 10, txt="Jupyter Notebook Export", ln=True)
C:\Users\balaj\AppData\Local\Temp\ipykernel_18760\3984464465.py:18: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, txt="Jupyter Notebook Export", ln=True)
C:\Users\balaj\AppData\Local\Temp\ipykernel_18760\3984464465.py:20: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=10)
C:\Users\balaj\AppData\Local\Temp\ipykernel_18760\3984464465.py:26: DeprecationWarni

In [ ]:
import json
import os
from fpdf import FPDF

# Load notebook
with open("main.ipynb", 'r', encoding='utf-8') as f:
    notebook = json.load(f)

# Create PDF
pdf = FPDF(orientation='P', format='A4')
pdf.set_auto_page_break(auto=True, margin=10)
pdf.add_page()

# Title
pdf.set_font("Helvetica", "B", size=16)
pdf.cell(0, 12, text="RAG FAQ Project - Jupyter Notebook Report", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", size=9)
pdf.cell(0, 6, text=f"Generated: {os.path.basename('main.ipynb')}", new_x="LMARGIN", new_y="NEXT")
pdf.ln(5)

# Process each cell
cell_count = 0
for cell in notebook['cells']:
    if pdf.get_y() > 270:
        pdf.add_page()
    
    cell_count += 1
    cell_type = cell.get('cell_type', 'unknown')
    
    # Cell header
    pdf.set_font("Helvetica", "B", size=10)
    if cell_type == 'code':
        pdf.set_text_color(200, 0, 0)
        pdf.cell(0, 5, text=f">>> CODE CELL {cell_count}", new_x="LMARGIN", new_y="NEXT")
    else:
        pdf.set_text_color(0, 0, 150)
        pdf.cell(0, 5, text=f"[MARKDOWN CELL {cell_count}]", new_x="LMARGIN", new_y="NEXT")
    
    pdf.set_text_color(0, 0, 0)
    pdf.ln(1)
    
    # Source code
    source = ''.join(cell.get('source', []))
    if source.strip():
        pdf.set_font("Courier", size=8)
        pdf.set_fill_color(240, 240, 240)
        
        for line in source.split('\n'):
            y_before = pdf.get_y()
            if len(line) > 90:
                line = line[:87] + "..."
            try:
                pdf.cell(0, 4, text=line, new_x="LMARGIN", new_y="NEXT", fill=True, border=0)
            except:
                pdf.ln(4)
    
    pdf.ln(1)
    
    # Cell outputs
    if cell_type == 'code' and cell.get('outputs'):
        pdf.set_font("Helvetica", "B", size=9)
        pdf.set_text_color(0, 100, 0)
        pdf.cell(0, 4, text="OUTPUT:", new_x="LMARGIN", new_y="NEXT")
        pdf.set_text_color(0, 0, 0)
        
        for output in cell['outputs']:
            output_type = output.get('output_type', '')
            
            if output_type == 'stream' and output.get('text'):
                pdf.set_font("Courier", size=7)
                pdf.set_fill_color(250, 250, 250)
                text = ''.join(output['text'])
                
                for line in text.split('\n'):
                    if line.strip():
                        if len(line) > 90:
                            line = line[:87] + "..."
                        try:
                            pdf.cell(0, 3.5, text=line, new_x="LMARGIN", new_y="NEXT", fill=True, border=0)
                        except:
                            pdf.ln(3.5)
            
            elif output_type == 'execute_result' and 'text/plain' in output.get('data', {}):
                pdf.set_font("Courier", size=7)
                pdf.set_fill_color(250, 250, 250)
                text_list = output['data']['text/plain']
                text = '\n'.join(text_list) if isinstance(text_list, list) else str(text_list)
                
                for line in text.split('\n'):
                    if line.strip():
                        if len(line) > 90:
                            line = line[:87] + "..."
                        try:
                            pdf.cell(0, 3.5, text=line, new_x="LMARGIN", new_y="NEXT", fill=True, border=0)
                        except:
                            pdf.ln(3.5)
            
            elif output_type == 'error':
                pdf.set_font("Courier", size=7)
                pdf.set_text_color(200, 0, 0)
                error = f"{output.get('ename', 'Error')}: {output.get('evalue', '')}"
                if len(error) > 90:
                    error = error[:87] + "..."
                pdf.cell(0, 3.5, text=error, new_x="LMARGIN", new_y="NEXT", fill=False)
                pdf.set_text_color(0, 0, 0)
        
        pdf.ln(2)

# Save
pdf.output("RAG_Notebook_Report.pdf")
print("=== PDF SUCCESSFULLY CREATED ===")
print(f"File: RAG_Notebook_Report.pdf")
print(f"Path: {os.path.abspath('RAG_Notebook_Report.pdf')}")
print(f"Cells: {cell_count}")
print("")
print("This PDF contains:")
print("  + All code cells with source")
print("  + All markdown cells")
print("  + Complete outputs from each cell")
print("  + Professional formatting")

SUCCESS! PDF generated successfully!

File Details:
  Name: notebook.pdf
  Path: c:\Users\balaj\OneDrive\Documents\vs\rag_faq_project\notebook.pdf
  Cells: 16
  Format: Landscape (A4)

Contents:
  ✓ All code cells
  ✓ All markdown cells
  ✓ Complete outputs
  ✓ Proper formatting
